In [1]:
# --- imports ---

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

from transformers import AutoModelForCausalLM, AutoTokenizer

In [45]:
# --- constants ---

MODEL_NAME = "Qwen/Qwen3.5-4B"
TOP_K = 10
DEVICE = "cuda"
DTYPE = torch.bfloat16

N_BEHAVIORAL_SAMPLES = 10
BEHAVIORAL_TEMPERATURE = 1.0
MAX_NEW_TOKENS = 300

In [3]:
# --- load Qwen ---

hf = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
    device_map="auto",
)
hf.eval()

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

print("loaded")
print(f"{torch.cuda.memory_allocated() / 2**30:.2f} GiB VRAM")
print("parameter devices:", {p.device for p in hf.parameters()})

In [ ]:
# --- non-experimental chat sanity check ---

messages = [
    {
        "role": "user",
        "content": "Give me three interesting facts about octopuses."
    }
]

inputs = tok.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(hf.device)

print("Rendered prompt:")
print(repr(tok.decode(inputs["input_ids"][0])))

with torch.inference_mode():
    outputs = hf.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
    )

generated = outputs[0, inputs["input_ids"].shape[1]:]

response = tok.decode(
    generated,
    skip_special_tokens=True,
)

print("\nResponse:")
print(response)

In [ ]:
def chat(prompt, max_new_tokens=300):
    messages = [
        {"role": "user", "content": prompt}
    ]

    inputs = tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    ).to(hf.device)

    with torch.inference_mode():
        outputs = hf.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=1.0,
            )

    generated = outputs[0, inputs["input_ids"].shape[1]:]

    return tok.decode(
        generated,
        skip_special_tokens=True,
    )